# Tome 5, Chapter 4: Dirichlet's theorem (Python companion)

Exercises first, solutions at the end. We sieve the primes up to $10^7$, split them into classes, check Theorem 4.1 and its corollaries, run the prime races of Figure 4.2, look at Dirichlet's own formulation with $p^{-\sigma}$, and use Dirichlet's theorem in a computer-science setting: primes $c\cdot2^k+1$ for the number-theoretic transform. Requirements: numpy, mpmath, matplotlib.

*Before each solution, predict what the output will look like.*

## Exercises

**P4.1 (★) Classes.** For $q=3,4,5,8,10,12$, count the primes up to $10^7$ in each class prime to $q$, and compute $R_a(x)=\sum_{p\le x,\,p\equiv a}\frac{\log p}p-\frac{\log x}{\varphi(q)}$ (bounded by Theorem 4.1).

**P4.2 (★) Constants.** Estimate $M(4,1)$ and $M(4,3)$ of Corollary 4.5 and check Exercise 4.2. Estimate the constants $C(4,1)$ and $C(4,3)$ of Exercise 4.4.

**P4.3 (★★) Prime races.** For several pairs of classes, compute $\pi(x;q,a)-\pi(x;q,b)$ up to $10^7$, the proportion of $[100,10^7]$ (in logarithmic scale) where $a$ leads, and the first $x$ where $a$ falls behind. Plot the normalised race $(\pi(x;4,3)-\pi(x;4,1))\frac{\log x}{\sqrt x}$.

**P4.4 (★) Dirichlet's formulation.** Compare $\sum_{p\equiv1(4)}p^{-\sigma}-\sum_{p\equiv3(4)}p^{-\sigma}$ with $\log L(\sigma,\chi_{-4})$ as $\sigma\to1^+$ (Exercise 4.3). Where does the difference come from?

**P4.5 (★★) CS application: primes for the number-theoretic transform.** An FFT modulo a prime $p$ of length $2^k$ needs a $2^k$-th root of unity in $\mathbb F_p$, that is $p\equiv1\pmod{2^k}$ (Tome 4, Chapter 7). Dirichlet's theorem guarantees infinitely many such primes. Find primes $c\cdot2^k+1$ for $k=16,20,23,26$, check that $998244353=119\cdot2^{23}+1$ is prime with primitive root $3$, and multiply two polynomials of degree $3000$ exactly with an NTT.

## Solutions

### P4.1

In [ ]:
import math
import numpy as np

N = 10**7
sieve = np.ones(N + 1, dtype=bool); sieve[:2] = False
for i in range(2, math.isqrt(N) + 1):
    if sieve[i]:
        sieve[i * i::i] = False
P = np.flatnonzero(sieve)
w = np.log(P) / P
print(f"{len(P)} primes up to {N:.0e}")
for q in (3, 4, 5, 8, 10, 12):
    units = [a for a in range(q) if math.gcd(a, q) == 1]
    phi = len(units)
    row = []
    for a in units:
        sel = P % q == a
        row.append((a, int(sel.sum()), w[sel].sum() - math.log(N) / phi))
    print(f"q = {q:2d}: " + "  ".join(f"[{a}] pi = {c}, R = {r:+.3f}" for a, c, r in row))

The counts in the classes agree to within $0.2\%$, and every $R_a(10^7)$ is between $-1.2$ and $0$, bounded as predicted. The constants differ between classes, systematically: classes containing squares (like $1$) have smaller $R_a$, because the prime powers $p^2\equiv1$ that $\Lambda$ would count are left out, and the values $\frac{L'}L(1,\chi)$ add their own offsets (Figure 4.1). For $q=10$ the table is that of $q=5$ in disguise: the classes modulo $10$ prime to $10$ correspond to the classes modulo $5$, apart from the prime $5$ itself.

### P4.2

In [ ]:
import mpmath as mp
gamma = float(mp.euler)
M_true = 0.2614972128476428
inv = 1.0 / P
loglogN = math.log(math.log(N))
M41 = inv[P % 4 == 1].sum() - loglogN / 2
M43 = inv[P % 4 == 3].sum() - loglogN / 2
print(f"M(4,1) ~ {M41:.5f}, M(4,3) ~ {M43:.5f}, sum = {M41 + M43:.5f}, M - 1/2 = {M_true - 0.5:.5f}")
for a in (1, 3):
    sel = P % 4 == a
    prod = np.exp(np.log1p(-inv[sel]).sum())
    print(f"a = {a}: prod (1 - 1/p) * (log x)^(1/2) = {prod * math.sqrt(math.log(N)):.5f}")

$M(4,1)\approx-0.2867$ and $M(4,3)\approx0.0482$ add up to $M-\frac12$ to five digits, as Exercise 4.2 says. The product constants are $C(4,1)\approx1.292$ and $C(4,3)\approx0.869$ (still drifting slowly at $10^7$, with errors $O(\frac1{\log x})$): the primes $\equiv3$ are slightly more numerous and come earlier ($3,7,11,19,\dots$ against $5,13,17,\dots$), so they sieve more.

### P4.3

In [ ]:
def race(q, a, b, X):
    """Return the running difference pi(x;q,a) - pi(x;q,b) at each prime up to X, and the primes."""
    Q = P[P <= X]
    steps = np.where(Q % q == a, 1, np.where(Q % q == b, -1, 0))
    return Q, np.cumsum(steps)

for (q, a, b) in ((4, 3, 1), (3, 2, 1), (8, 3, 1), (8, 5, 1), (8, 7, 1), (10, 3, 1)):
    Q, D = race(q, a, b, N)
    lead = D > 0
    # logarithmic share of the "time" x in [2, N] where class a strictly leads
    edges = np.append(Q, N)
    seg = np.log(edges[1:]) - np.log(edges[:-1])
    keep = Q >= 100                                   # measure on [100, N], away from the small primes
    share = seg[lead & keep].sum() / seg[keep].sum()
    first = Q[np.argmax(D < 0)] if (D < 0).any() else None
    print(f"q = {q:2d}: class {a} vs {b}: leads on {100 * share:6.2f}% of [100, 1e7] (log scale), first time behind: {first}, final lead {D[-1]}")

In [ ]:
import matplotlib.pyplot as plt
Q, D = race(4, 3, 1, N)
fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(Q[::50], D[::50] / np.sqrt(Q[::50]) * np.log(Q[::50]), lw=0.6)
ax.axhline(0, color="gray", lw=0.5)
ax.set_xscale("log"); ax.set_xlabel("x"); ax.set_ylabel("(pi(x;4,3) - pi(x;4,1)) log x / sqrt x")
plt.show()

Every race here is led by the non-square class: $3$ against $1$ modulo $4$, $2$ against $1$ modulo $3$ (the first lead change is near $6\cdot10^{11}$, found by Bays and Hudson), $3,5,7$ against $1$ modulo $8$. Modulo $4$, class $1$ leads only at $x=26861$ below $10^7$, and the share $99.88\%$ on $[100,10^7]$ is close to the Rubinstein–Sarnak density $99.59\%$. The normalised plot oscillates around a positive mean: this is the bias $\frac{\sqrt x}{\log x}$ of the remark, with fluctuations of the same size driven by the zeros of $L(s,\chi_{-4})$ (Chapter 7).

### P4.4

In [ ]:
def L_hurwitz(s, chi, q):
    return mp.mpf(q)**(-s) * sum(chi[a] * mp.zeta(s, mp.mpf(a) / q) for a in range(1, q) if chi[a] != 0)

chi4 = [0, 1, 0, -1]
Pf = P.astype(float)
for sigma in (1.5, 1.1, 1.01):
    S1 = (Pf[P % 4 == 1] ** -sigma).sum()
    S3 = (Pf[P % 4 == 3] ** -sigma).sum()
    S2 = 2.0 ** -sigma
    print(f"sigma = {sigma}: sum_(p=1) - sum_(p=3) of p^-s (p <= 1e7) = {S1 - S3:+.4f}  vs  log L(sigma, chi_4) = {float(mp.log(L_hurwitz(sigma, chi4, 4))):+.4f};"
          f"  class 1 share of sum_(p odd) p^-s = {S1 / (S1 + S3):.4f}")

The difference of the prime sums tracks $\log L(\sigma,\chi_{-4})$ up to a bounded discrepancy: by Theorem 3.6(a), $\log L(\sigma,\chi)=\sum_p\chi(p)p^{-\sigma}+\sum_p\sum_{k\ge2}\frac{\chi(p)^k}{kp^{k\sigma}}$, and the $k=2$ terms are all positive (about $0.09$ here). Both sides stay bounded as $\sigma\to1^+$ because $L(1,\chi_{-4})=\frac\pi4\ne0$, while each prime sum alone grows like $\frac12\log\frac1{\sigma-1}$: the share of class $1$ creeps towards $\frac12$ ($0.431$ at $\sigma=1.01$; the truncation at $10^7$ also matters there).

### P4.5

In [ ]:
def is_prime(n):
    if n < 2:
        return False
    for p in (2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37):
        if n % p == 0:
            return n == p
    d, s = n - 1, 0
    while d % 2 == 0:
        d //= 2; s += 1
    for a in (2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37):
        x = pow(a, d, n)
        if x in (1, n - 1):
            continue
        for _ in range(s - 1):
            x = x * x % n
            if x == n - 1:
                break
        else:
            return False
    return True

for k in (16, 20, 23, 26):
    cs = [c for c in range(1, 2000) if is_prime(c * 2**k + 1)][:6]
    print(f"primes c * 2^{k} + 1, c < 2000: c = {cs}")
p = 998244353
print(p, "=", (p - 1) // 2**23, "* 2^23 + 1;  prime:", is_prime(p))

In [ ]:
def ntt(a, p, g, invert=False):
    """Iterative number-theoretic transform modulo p (length a power of 2 dividing p - 1), g a primitive root."""
    n = len(a); a = a[:]
    j = 0
    for i in range(1, n):
        bit = n >> 1
        while j & bit:
            j ^= bit; bit >>= 1
        j |= bit
        if i < j:
            a[i], a[j] = a[j], a[i]
    length = 2
    while length <= n:
        w = pow(g, (p - 1) // length, p)
        if invert:
            w = pow(w, p - 2, p)
        for i in range(0, n, length):
            wn = 1
            for k in range(length // 2):
                u, v = a[i + k], a[i + k + length // 2] * wn % p
                a[i + k], a[i + k + length // 2] = (u + v) % p, (u - v) % p
                wn = wn * w % p
        length <<= 1
    if invert:
        ninv = pow(n, p - 2, p)
        a = [x * ninv % p for x in a]
    return a

def multiply(x, y, p=998244353, g=3):
    n = 1
    while n < len(x) + len(y):
        n <<= 1
    fx, fy = ntt(x + [0] * (n - len(x)), p, g), ntt(y + [0] * (n - len(y)), p, g)
    return ntt([a * b % p for a, b in zip(fx, fy)], p, g, invert=True)[:len(x) + len(y) - 1]

import random
random.seed(2)
x = [random.randrange(100) for _ in range(3000)]
y = [random.randrange(100) for _ in range(3000)]
exact = np.convolve(np.array(x, dtype=object), np.array(y, dtype=object))
print("NTT product equals the exact convolution:", multiply(x, y) == [int(v) for v in exact])
print("3 is a primitive root mod 998244353:", all(pow(3, (p - 1) // r, p) != 1 for r in (2, 7, 17)))

Primes $c\cdot2^k+1$ are plentiful: by Theorem 4.1 (with the prime number theorem for progressions), about $\frac1{\varphi(2^k)}$ of all primes, and among the numbers $c\cdot2^k+1$ with $c\le C$ we expect about $\frac{C}{\log(C2^k)}\cdot\frac{2^k}{\varphi(2^k)}=\frac{2C}{\log(C2^k)}$ primes. The prime $998244353=119\cdot2^{23}+1$, with primitive root $3$, is the standard NTT modulus of competitive programming: it supports transforms of length up to $2^{23}$ with exact integer arithmetic, and the product computed modulo $p$ equals the true product when every true coefficient is below $p$: here they are at most $3000\cdot99^2<3\cdot10^7$. For larger inputs one uses two or three NTT primes and the Chinese remainder theorem.